# Group XX - Machine Learning, Element 1: Apply dataset preparation mechanisms
| Item | Details |
|---|---|
| Group | **Group XX** |
| Members | **IGIHOZO Bienvenue** and **MUGISHA Aime Patrick** |
| Question | **Q1 - Predicting maize yield for farmer cooperatives in Rwanda (MCSU)** |
| Stage | **2 - Define requirements and acquire data** |
| Driver / Navigator | Driver: MUGISHA Aime Patrick / Navigator: IGIHOZO Bienvenue |


## Stage 2 - Define requirements and acquire data (PC 1.2)

### 2.1 Data requirements (from Mr. Habimana's message and the five business questions)
| Requirement | Decision |
|---|---|
| **Target** | maize yield per hectare - `yield_t_ha` in the file (we also give `yield_kg_ha`). For question E: `low_yield_risk` = bottom 10 % of training yields |
| **Unit of observation** | one farm in one season |
| **Features needed** | location (province, district, altitude), season and year, farm area, seed variety, farming practice, fertiliser **per hectare** (NPK, urea, manure), plant density, rainfall, temperature, soil type and pH, pest level, irrigation |
| **Must NOT be used** | anything known only after harvest (harvest weight, sale price...) - leakage (question E) |
| **Time period / area** | Seasons A, B, C over several years; all provinces |
| **Records needed** | at least ~1,000 usable farm-season records so that train/val/test (70/15/15) keep every season |
| **Quality** | one label per category, one unit per measure, realistic values, no duplicates |

### 2.2 Which data serves which business question
| Q | Needs | Columns in the file |
|---|---|---|
| A | reliable yield per farm | `yield_t_ha`, `farm_area_ha`, `plant_density_per_ha` |
| B | fair fertiliser comparison, correct seed | `npk_kg_ha`, `urea_kg_ha`, `manure_t_ha` (already per ha), `seed_variety`, `farming_practice` |
| C | weather, altitude, irrigation | `rainfall_mm`, `avg_temp_c`, `altitude_m`, `irrigation`, `soil_ph`, `soil_type` |
| D | consistent province/district/season | `province`, `district`, `season`, `year` |
| E | only pre-harvest information | all except the target (see timing note in Stage 3) |

In [1]:
import sys  # lets us tell Python where our own code (src/) is
from pathlib import Path  # file paths that work on every computer
import warnings  # hide harmless library notices
warnings.filterwarnings("ignore")  # keep the output readable
import numpy as np  # maths on whole columns
import pandas as pd  # the DataFrame table
ROOT = Path.cwd().parent  # notebooks/ is inside q1_maize_yield
sys.path.insert(0, str(ROOT / "src"))  # make src/maize_pipeline.py importable
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)  # show every column
RAW = ROOT / "data" / "raw" / "maize_yield_rwanda_rw.csv"  # the original file (never edited)
import maize_pipeline as mp  # our reusable functions

### 2.3 Acquire the data
The file `maize_yield_rwanda_rw.csv` supplied by the lecturer is copied **unchanged** into `data/raw/`
(SHA-256 identical to the original). We only read it; every change happens later in code.

In [2]:
raw = pd.read_csv(RAW)  # read the CSV into a DataFrame
print("File :", RAW.name)  # file name
print("Shape:", raw.shape, "-> rows, columns")  # size
raw.head()  # first 5 records

File : maize_yield_rwanda_rw.csv
Shape: (1426, 20) -> rows, columns


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
0,RW-MZ-00001,2015,A,Eastern,Bugesera,1349,NaN,22.9,NaN,Clay,RHM104,Improved,117.8,57.4,1.41,61803,0.63,No,NaN,2.71
1,RW-MZ-00002,2015,A,Eastern,Bugesera,1427,489.8,22.1,6.02,Volcanic,RHM104,Improved,38.1,62.7,3.69,52919,0.17,No,0.339,4.09
2,RW-MZ-00003,2015,A,Northern,Burera,2033,722.6,16.3,5.73,Clay loam,Local,Improved,121.1,50.7,5.18,54406,0.22,No,0.049,2.58
3,RW-MZ-00004,2015,A,Northern,Burera,2026,794.4,16.4,5.20,Clay loam,ZM621,Improved,93.7,60.4,1.68,48119,0.48,No,0.171,2.45
4,RW-MZ-00005,2015,A,Northern,Gakenke,1974,644.2,18.3,5.92,Loam,Local,Improved,84.1,70.4,0.36,56787,1.16,No,0.001,2.09


In [3]:
raw.tail()  # last 5 records - the whole file was read

,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
1421,RW-MZ-01422,2024,C,Southern,Nyanza,1748,228.6,19.3,5.96,Clay,ZM621,Traditional,9.1,13.6,2.86,36249,1.54,No,0.284,1.89
1422,RW-MZ-01423,2024,C,Northern,Rulindo,1881,211.5,18.4,5.45,Volcanic,ZM607,Improved,36.3,66.0,1.64,64018,0.34,No,0.403,2.21
1423,RW-MZ-01424,2024,C,Northern,Rulindo,1887,242.3,18.2,5.70,Volcanic,WH505,Traditional,7.4,37.7,1.29,24474,1.95,No,0.304,2.10
1424,RW-MZ-01425,2024,C,Western,Rusizi,1409,310.5,19.6,5.02,Sandy loam,ZM607,Traditional,13.7,0.0,0.55,43111,0.20,No,0.542,1.40
1425,RW-MZ-01426,2024,C,Western,Rusizi,1583,391.0,20.5,5.01,Sandy loam,ZM607,Improved,86.7,50.8,2.50,54197,0.77,No,0.193,2.12


In [4]:
raw.info()  # column names, non-empty counts and data types

<class 'pandas.DataFrame'>
RangeIndex: 1426 entries, 0 to 1425
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   record_id             1426 non-null   str    
 1   year                  1426 non-null   int64  
 2   season                1426 non-null   str    
 3   province              1426 non-null   str    
 4   district              1426 non-null   str    
 5   altitude_m            1426 non-null   int64  
 6   rainfall_mm           1415 non-null   float64
 7   avg_temp_c            1426 non-null   float64
 8   soil_ph               1412 non-null   float64
 9   soil_type             1426 non-null   str    
 10  seed_variety          1426 non-null   str    
 11  farming_practice      1426 non-null   str    
 12  npk_kg_ha             1426 non-null   float64
 13  urea_kg_ha            1426 non-null   float64
 14  manure_t_ha           1426 non-null   float64
 15  plant_density_per_ha  1426 non-n

**First observations:** 1,426 rows x 20 columns. Numbers are already numeric and text columns are text; only
`rainfall_mm`, `soil_ph` and `pest_incidence_index` have empty cells.

### 2.4 Check: does the file meet the requirements?

In [5]:
check = pd.DataFrame({  # requirement -> evidence from the file
    "requirement": ["at least 1,000 records", "target present", "seasons A, B, C", "all provinces",
                    "years covered", "fertiliser per hectare", "no post-harvest column"],
    "evidence": [len(raw), mp.TARGET in raw.columns, sorted(raw["season"].unique()), sorted(raw["province"].unique()),
                 f"{raw['year'].min()}-{raw['year'].max()}", [c for c in raw.columns if c.endswith("_ha")],
                 "none found (no harvest weight, sales or price)"],
})
check  # show

,requirement,evidence
0,"at least 1,000 records",1426
1,target present,True
2,"seasons A, B, C","[A, B, C]"
3,all provinces,"[Eastern, Kigali, Northern, Southern, Western]"
4,years covered,2015-2024
5,fertiliser per hectare,"[npk_kg_ha, urea_kg_ha, manure_t_ha, plant_den..."
6,no post-harvest column,"none found (no harvest weight, sales or price)"


**Conclusion:** the file covers all five questions: yield per hectare (A), fertiliser already **per hectare** plus seed and
practice (B), weather/altitude/irrigation (C), 5 provinces and 30 districts over 2015-2024 (D) and no post-harvest
column (E). Differences from the brief: yield is in **t/ha** (not kg/ha), fertiliser is **per ha** (not whole-farm), and there
is no farm/cooperative code or planting date - so questions about individual cooperatives or planting dates cannot be answered.

### 2.5 Data dictionary (Appendix B) - saved to `docs/data_dictionary.csv`

In [6]:
dictionary = pd.DataFrame([  # column, meaning, type, unit, allowed values
    ("record_id", "Record code", "text (ID)", "-", "RW-MZ-00001 ... unique; not a feature"),
    ("year", "Year of the season", "integer", "year", "2015-2024"),
    ("season", "Agricultural season", "category", "-", "A (Sep-Jan), B (Feb-Jun), C (Jun-Sep)"),
    ("province", "Province", "category", "-", "Kigali City, Eastern, Northern, Southern, Western"),
    ("district", "District", "category", "-", "30 official districts"),
    ("altitude_m", "Altitude of the farm", "integer", "m", "900-2,600 in Rwanda"),
    ("rainfall_mm", "Rainfall during the season", "float", "mm", "> 0"),
    ("avg_temp_c", "Average season temperature", "float", "deg C", "maize best 18-27"),
    ("soil_ph", "Soil acidity", "float", "pH", "4-8 normal; maize best 5.5-7"),
    ("soil_type", "Soil type", "category", "-", "Clay, Clay loam, Loam, Sandy loam, Volcanic"),
    ("seed_variety", "Seed variety", "category", "-", "Local, ZM607, ZM621 (OPV), RHM104, WH505, PAN691 (hybrids)"),
    ("farming_practice", "Farming practice", "category", "-", "Improved / Traditional"),
    ("npk_kg_ha", "NPK fertiliser per hectare", "float", "kg/ha", ">= 0 (0 = none used)"),
    ("urea_kg_ha", "Urea fertiliser per hectare", "float", "kg/ha", ">= 0"),
    ("manure_t_ha", "Organic manure per hectare", "float", "t/ha", ">= 0"),
    ("plant_density_per_ha", "Maize plants per hectare", "integer", "plants/ha", "about 20,000-75,000"),
    ("farm_area_ha", "Area of the maize farm", "float", "ha", "> 0"),
    ("irrigation", "Plot irrigated", "category", "-", "Yes / No"),
    ("pest_incidence_index", "Pest pressure", "float", "0-1 index", "0 = none, 1 = severe"),
    ("yield_t_ha", "Maize yield - TARGET", "float", "t/ha", "> 0; typical 1-4 t/ha"),
], columns=["column", "meaning", "data_type", "unit", "allowed_values_notes"])
(ROOT / "docs").mkdir(exist_ok=True)  # make sure docs/ exists
dictionary.to_csv(ROOT / "docs" / "data_dictionary.csv", index=False)  # save
dictionary  # display

,column,meaning,data_type,unit,allowed_values_notes
0,record_id,Record code,text (ID),-,RW-MZ-00001 ... unique; not a feature
1,year,Year of the season,integer,year,2015-2024
2,season,Agricultural season,category,-,"A (Sep-Jan), B (Feb-Jun), C (Jun-Sep)"
3,province,Province,category,-,"Kigali City, Eastern, Northern, Southern, Western"
4,district,District,category,-,30 official districts
5,altitude_m,Altitude of the farm,integer,m,"900-2,600 in Rwanda"
6,rainfall_mm,Rainfall during the season,float,mm,> 0
7,avg_temp_c,Average season temperature,float,deg C,maize best 18-27
8,soil_ph,Soil acidity,float,pH,4-8 normal; maize best 5.5-7
9,soil_type,Soil type,category,-,"Clay, Clay loam, Loam, Sandy loam, Volcanic"


### 2.6 Source log - saved to `docs/source_log.csv`

In [7]:
source_log = pd.DataFrame([{  # where the data came from
    "source": "File maize_yield_rwanda_rw.csv supplied by the lecturer for Question 1 (MCSU case study)",
    "link": "data/raw/maize_yield_rwanda_rw.csv (course material)",
    "date_accessed": "2026-11-11",
    "licence": "course material - for this course only",
    "collected_by": "MCSU field officers (case study); weather values per season",
    "limitations": "no farm/cooperative code or planting date; yield in t/ha; Season C and irrigated farms are few; rainfall is a season total",
    "ethical_issues": "no names, phones or IDs of farmers; possible bias: equal numbers per district may not reflect real maize area",
}])
source_log.to_csv(ROOT / "docs" / "source_log.csv", index=False)  # save
source_log.T  # display vertically

,0
source,File maize_yield_rwanda_rw.csv supplied by the...
link,data/raw/maize_yield_rwanda_rw.csv (course mat...
date_accessed,2026-11-11
licence,course material - for this course only
collected_by,MCSU field officers (case study); weather valu...
limitations,no farm/cooperative code or planting date; yie...
ethical_issues,"no names, phones or IDs of farmers; possible b..."
